In [10]:
import os

files = [file for file in os.listdir("data/") if file.endswith(".pgn")]

In [11]:
len(files)

4

In [12]:
from chess import pgn

def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games

In [13]:
games = []
games.extend(load_pgn(f"data/lichess_elite_2025-02.pgn"))

In [14]:
len(games)

256430

In [15]:
import numpy as np
from chess import Board
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam # type: ignore
import time

In [16]:
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix


def create_input_for_nn(games):
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y


def encode_moves(moves):
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

In [17]:
games_short = games[:5000]

In [18]:
X, y = create_input_for_nn(games_short)
y, move_to_int = encode_moves(y)
y = to_categorical(y, num_classes=len(move_to_int))
X = np.array(X)

In [22]:
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(8, 8, 12)),
    Conv2D(128, (3, 3), activation='relu'),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(len(move_to_int), activation='softmax')
])
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X, y, epochs=50, validation_split=0.2, batch_size=64)
model.save("models/TF_50EPOCHS.keras")

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 6, 6, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1846)           │       474,422 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,079,798 (4.12 MB)

 Trainable params: 1,079,798 (4.12 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 30s 5ms/step - accuracy: 0.0615 - loss: 6.0427 - val_accuracy: 0.1216 - val_loss: 4.9935
Epoch 2/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 30s 5ms/step - accuracy: 0.1440 - loss: 4.6162 - val_accuracy: 0.1422 - val_loss: 4.5297
Epoch 3/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 30s 6ms/step - accuracy: 0.1738 - loss: 4.0857 - val_accuracy: 0.1526 - val_loss: 4.3953
Epoch 4/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.1960 - loss: 3.8015 - val_accuracy: 0.1587 - val_loss: 4.3450
Epoch 5/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.2111 - loss: 3.6219 - val_accuracy: 0.1605 - val_loss: 4.3758
Epoch 6/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 30s 5ms/step - accuracy: 0.2280 - loss: 3.4772 - val_accuracy: 0.1589 - val_loss: 4.3950
Epoch 7/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 32s 6ms/step - accuracy: 0.2423 - loss: 3.3568 - val_accuracy: 0.1596 - val_loss: 4.4358
Epoch 8/50
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 30s 6ms/step - accuracy: 0.2532 - loss: 3

In [193]:
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(8, 8, 12)),
    Conv2D(128, (3, 3), activation='relu'),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(len(move_to_int), activation='softmax')
])
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X, y, epochs=10, validation_split=0.2, batch_size=64)
model.save("models/TF_10EPOCHS.keras")

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 6, 6, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1846)           │       474,422 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,079,798 (4.12 MB)

 Trainable params: 1,079,798 (4.12 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - accuracy: 0.0611 - loss: 6.0404 - val_accuracy: 0.1218 - val_loss: 4.9920
Epoch 2/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.1427 - loss: 4.6273 - val_accuracy: 0.1445 - val_loss: 4.5385
Epoch 3/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 28s 5ms/step - accuracy: 0.1718 - loss: 4.0924 - val_accuracy: 0.1539 - val_loss: 4.4222
Epoch 4/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.1955 - loss: 3.7981 - val_accuracy: 0.1526 - val_loss: 4.3589
Epoch 5/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.2139 - loss: 3.6106 - val_accuracy: 0.1586 - val_loss: 4.3705
Epoch 6/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 28s 5ms/step - accuracy: 0.2307 - loss: 3.4570 - val_accuracy: 0.1593 - val_loss: 4.3991
Epoch 7/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.2428 - loss: 3.3513 - val_accuracy: 0.1582 - val_loss: 4.4433
Epoch 8/10
5404/5404 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.2564 - loss: 3

In [24]:
# Load the trained model
from tensorflow.keras.models import load_model
model = load_model("models/TF_50EPOCHS.keras")

In [25]:
int_to_move = dict(zip(move_to_int.values(), move_to_int.keys()))


def predict_next_move(board):
    board_matrix = board_to_matrix(board).reshape(1, 8, 8, 12)
    predictions = model.predict(board_matrix)[0]
    legal_moves = list(board.legal_moves)
    legal_moves_uci = [move.uci() for move in legal_moves]
    sorted_indices = np.argsort(predictions)[::-1]
    for move_index in sorted_indices:
        move = int_to_move[move_index]
        if move in legal_moves_uci:
            return move
    return None

In [26]:
# Create a chess board (start position)
board = Board()

In [191]:
# Display the board before prediction
print("Board before prediction:")
print(board)

# Predict and make the move
next_move = predict_next_move(board)
board.push_uci(next_move)

# Display the board after prediction
print("\nPredicted move:", next_move)
print("Board after prediction:")
print(board)

Board before prediction:
. . . . . . . k
. . . . . . . .
. . . . . . . P
. . . . . . . .
P K . . . . . .
. . . . . . . q
. . . . . . . .
. . . . . . . R
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

Predicted move: h1g1
Board after prediction:
. . . . . . . k
. . . . . . . .
. . . . . . . P
. . . . . . . .
P K . . . . . .
. . . . . . . q
. . . . . . . .
. . . . . . R .


In [192]:
print(str(pgn.Game.from_board(board)))

[Event "?"]
[Site "?"]
[Date "????.??.??"]
[Round "?"]
[White "?"]
[Black "?"]
[Result "*"]

1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. Nc3 a6 6. Be3 e5 7. Nb3 Be6 8. f3 Nbd7 9. Qd2 b5 10. O-O-O Nb6 11. Kb1 Rc8 12. Bxb6 Qxb6 13. Nd5 Bxd5 14. exd5 h6 15. Qe3 a5 16. Be2 g6 17. Rc1 Rxc2 18. Rcd1 Rc4 19. Bxc4 a4 20. Be2 Qd8 21. Nc5 Be7 22. Rd4 dxc5 23. Rh4 Nxd5 24. Qd2 O-O 25. Qxd5 Qxd5 26. g3 c4 27. Kc2 Kg7 28. a3 f5 29. g4 Bf6 30. gxf5 gxf5 31. Rh5 Kf7 32. h4 Qe6 33. Bxc4 bxc4 34. Rxh6 e4 35. fxe4 fxe4 36. Rh7+ Ke8 37. Rd7 Qf5 38. h5 Bd4 39. h6 Bxb2 40. Kxb2 Qg6 41. Kc3 Qg4 42. Rg1 Rg8 43. Rg7 Rh8 44. Rf7 Kd8 45. Rh7 Re8 46. Rh8 Rf8 47. Rh7 Qf4 48. Rh8 Qf5 49. Rhg8 Re8 50. R8g4 e3 51. Rh1 Re4 52. Rg2 Ke8 53. Rg7 Qh5 54. Rhg1 Rf4 55. Kb4 Qg5 56. Rh7 Re4 57. Rh8+ Kf7 58. Rh7+ Kg8 59. Rh8+ Kxh8 60. Rh1 Qg6 61. Kxa4 Re6 62. Ka5 Rc6 63. Re1 Rd6 64. Rxe3 Rd2 65. Kb4 Rg2 66. Ka5 Rg1 67. Re4 c3 68. Rc4 c2 69. Rc3 Rc1 70. Kb4 Rd1 71. Rxc2 Qf5 72. Rh2 Rf1 73. a4 Rg1 74. Rh1 Qf1 75. Rh2 Qf3 76. R